In [25]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import average_precision_score
from tqdm import tqdm

In [26]:
CONFIG = {
    "seq_len": 6,
    "batch_size": 64,
    "lr": 1e-3,
    "epochs": 50,
    "patience": 10,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}
print("Using device:", CONFIG["device"])

Using device: cpu


In [27]:
TARGET = "deterioration_next_12h"

df_sample = pd.read_csv("../data/features/features.csv")

# REMOVE NON-NUMERIC SAFELY
EXCLUDE = ["patient_id", TARGET]

FEATURES = [
    col for col in df_sample.columns
    if col not in EXCLUDE and df_sample[col].dtype != "object"
]

NUM_FEATURES = len(FEATURES)

print("Number of features:", NUM_FEATURES)

Number of features: 58


In [28]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df, features, seq_len=6):
        self.X = []
        self.y = []
        
        for pid, group in df.groupby("patient_id"):
            group = group.sort_values("hour_from_admission")
            
            data = group[features].apply(pd.to_numeric, errors='coerce').values
            target = group[TARGET].values
            
            for i in range(len(group) - seq_len):
                self.X.append(data[i:i+seq_len])
                self.y.append(target[i+seq_len-1])
        
        self.X = np.array(self.X)
        self.X = np.nan_to_num(self.X)
        
        self.y = np.array(self.y)
        
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [29]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        pt = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

In [30]:
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        out = self.conv(x)
        out = out[:, :, :-self.conv.padding[0]]
        out = self.relu(out)
        return self.dropout(out)

In [34]:
class TCN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        
        self.tcn = nn.Sequential(
            TCNBlock(num_features, 64, dilation=1),
            TCNBlock(64, 64, dilation=2),
            TCNBlock(64, 64, dilation=4),
            TCNBlock(64, 64, dilation=8),
        )
        
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out = self.tcn(x)
        out = out[:, :, -1]
        return self.fc(out).squeeze(-1)

In [35]:
def train_model(train_loader, val_loader, fold):
    model = TCN(NUM_FEATURES).to(CONFIG["device"])
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
    criterion = FocalLoss()
    
    best_score = 0
    patience_counter = 0
    
    for epoch in range(CONFIG["epochs"]):
        model.train()
        
        for X, y in train_loader:
            X = X.to(CONFIG["device"])
            y = y.to(CONFIG["device"])
            
            optimizer.zero_grad()
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # Validation
        model.eval()
        preds, targets = [], []
        
        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(CONFIG["device"])
                outputs = torch.sigmoid(model(X)).cpu().numpy()
                
                preds.extend(outputs)
                targets.extend(y.numpy())
        
        pr_auc = average_precision_score(targets, preds)
        
        print(f"Fold {fold} | Epoch {epoch} | PR-AUC: {pr_auc:.4f}")
        
        if pr_auc > best_score:
            best_score = pr_auc
            patience_counter = 0
            torch.save(model.state_dict(), f"../models/tcn/best_tcn_fold_{fold}.pt")
        else:
            patience_counter += 1
        
        if patience_counter >= CONFIG["patience"]:
            print("Early stopping")
            break
    
    return best_score

In [36]:
os.makedirs("../models/tcn", exist_ok=True)
os.makedirs("../outputs/metrics", exist_ok=True)

scores = []

for fold in range(5):
    print(f"\n==== Fold {fold} ====")
    
    train_df = pd.read_csv(f"../data/splits/train_fold_{fold}.csv")
    val_df = pd.read_csv(f"../data/splits/val_fold_{fold}.csv")
    
    train_ds = TimeSeriesDataset(train_df, FEATURES, CONFIG["seq_len"])
    val_ds = TimeSeriesDataset(val_df, FEATURES, CONFIG["seq_len"])
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)
    
    score = train_model(train_loader, val_loader, fold)
    scores.append(score)


==== Fold 0 ====
Fold 0 | Epoch 0 | PR-AUC: 0.6051
Fold 0 | Epoch 1 | PR-AUC: 0.6292
Fold 0 | Epoch 2 | PR-AUC: 0.6153
Fold 0 | Epoch 3 | PR-AUC: 0.6478
Fold 0 | Epoch 4 | PR-AUC: 0.5828
Fold 0 | Epoch 5 | PR-AUC: 0.6496
Fold 0 | Epoch 6 | PR-AUC: 0.5680
Fold 0 | Epoch 7 | PR-AUC: 0.5914
Fold 0 | Epoch 8 | PR-AUC: 0.6235
Fold 0 | Epoch 9 | PR-AUC: 0.6466
Fold 0 | Epoch 10 | PR-AUC: 0.6464
Fold 0 | Epoch 11 | PR-AUC: 0.6572
Fold 0 | Epoch 12 | PR-AUC: 0.6001
Fold 0 | Epoch 13 | PR-AUC: 0.6284
Fold 0 | Epoch 14 | PR-AUC: 0.6418
Fold 0 | Epoch 15 | PR-AUC: 0.5563
Fold 0 | Epoch 16 | PR-AUC: 0.6433
Fold 0 | Epoch 17 | PR-AUC: 0.6398
Fold 0 | Epoch 18 | PR-AUC: 0.6241
Fold 0 | Epoch 19 | PR-AUC: 0.6344
Fold 0 | Epoch 20 | PR-AUC: 0.5621
Fold 0 | Epoch 21 | PR-AUC: 0.6491
Early stopping

==== Fold 1 ====
Fold 1 | Epoch 0 | PR-AUC: 0.5704
Fold 1 | Epoch 1 | PR-AUC: 0.6103
Fold 1 | Epoch 2 | PR-AUC: 0.6281
Fold 1 | Epoch 3 | PR-AUC: 0.6156
Fold 1 | Epoch 4 | PR-AUC: 0.5612
Fold 1 | Epoch 5 | 

In [37]:
scores = np.array(scores)

print("\nTCN CV Results:")
print("Mean PR-AUC:", scores.mean())
print("Std:", scores.std())

pd.DataFrame({"pr_auc": scores}).to_csv("../outputs/metrics/tcn_cv.csv", index=False)


TCN CV Results:
Mean PR-AUC: 0.6502222662827648
Std: 0.011318992378248382
